## prepare data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(
    r"../data/raw/english/sentiment_data.csv")

print(df.shape)

(241145, 3)


In [3]:
print(f"first 5 rows of df: {df.head()}")
print("================================================")
print(f"Data types and missing values: {df.info()}")
print("================================================")

print(f"Column names: {df.columns.tolist()}")


first 5 rows of df:    Unnamed: 0                                            Comment  Sentiment
0           0  lets forget apple pay required brand new iphon...          1
1           1  nz retailers don’t even contactless credit car...          0
2           2  forever acknowledge channel help lessons ideas...          2
3           3  whenever go place doesn’t take apple pay doesn...          0
4           4  apple pay convenient secure easy use used kore...          2
<class 'pandas.DataFrame'>
RangeIndex: 241145 entries, 0 to 241144
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   Unnamed: 0  241145 non-null  int64
 1   Comment     240928 non-null  str  
 2   Sentiment   241145 non-null  int64
dtypes: int64(2), str(1)
memory usage: 25.9 MB
Data types and missing values: None
Column names: ['Unnamed: 0', 'Comment', 'Sentiment']


In [4]:
df_model = df[["Comment", "Sentiment"]].copy()

df_model.rename(
    columns={
        "Comment": "text",
        "Sentiment": "sentiment"
    },
    inplace=True
)

In [5]:
label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

df_model["sentiment"] = df_model["sentiment"].map(label_map)

In [6]:
df_model = df_model.dropna(subset=["text"])

In [7]:
#clean texts
import re
import html

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Decode HTML entities
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove mentions
    text = re.sub(r"@\w+", "", text)

    # Remove RT
    text = re.sub(r"^RT\s+", "", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [8]:
df_model["text"] = df_model["text"].apply(clean_text)

In [9]:
df_model = df_model[df_model["text"].str.strip() != ""]

In [10]:
df_model = df_model.drop_duplicates(subset=["text"])

In [11]:
df_model.reset_index(drop=True, inplace=True)

In [12]:
print(df_model.isnull().sum())
print(df_model.duplicated().sum())
print(df_model["sentiment"].value_counts())

text         0
sentiment    0
dtype: int64
0
sentiment
positive    94195
neutral     71690
negative    48267
Name: count, dtype: int64


In [13]:
print(df_model["text"].str.len().describe())

count    214152.000000
mean         92.940603
std          73.315993
min           1.000000
25%          45.000000
50%          78.000000
75%         132.000000
max        5080.000000
Name: text, dtype: float64


In [14]:
df_model = df_model[df_model["text"].str.len() >= 10]

In [15]:
print(df_model["sentiment"].value_counts())

sentiment
positive    93790
neutral     70381
negative    47980
Name: count, dtype: int64


## Split data

In [16]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_model,
    test_size=0.2,
    stratify=df_model["sentiment"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["sentiment"],
    random_state=42
)

In [17]:
print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(169720, 2)
(21215, 2)
(21216, 2)


In [18]:
print(train_df["sentiment"].value_counts())
print(valid_df["sentiment"].value_counts())
print(test_df["sentiment"].value_counts())

sentiment
positive    75032
neutral     56304
negative    38384
Name: count, dtype: int64
sentiment
positive    9379
neutral     7038
negative    4798
Name: count, dtype: int64
sentiment
positive    9379
neutral     7039
negative    4798
Name: count, dtype: int64


In [19]:
train_df.to_csv("../data/processed/english/train.csv", index=False, encoding="utf-8-sig")

valid_df.to_csv("../data/processed/english/validation.csv", index=False, encoding="utf-8-sig")

test_df.to_csv("../data/processed/english/test.csv", index=False, encoding="utf-8-sig")

In [20]:
train = pd.read_csv("../data/processed/english/train.csv", keep_default_na=False)
valid = pd.read_csv("../data/processed/english/validation.csv", keep_default_na=False)
test = pd.read_csv("../data/processed/english/test.csv", keep_default_na=False)

print(train.isnull().sum())
print(valid.isnull().sum())
print(test.isnull().sum())

text         0
sentiment    0
dtype: int64
text         0
sentiment    0
dtype: int64
text         0
sentiment    0
dtype: int64
